In [8]:
!pip install pyspark -q

import os, json, random, sys
from datetime import datetime, timedelta

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

# Set Python executable untuk mengatasi version mismatch
python_exec = sys.executable
os.environ["PYSPARK_PYTHON"] = python_exec
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exec

spark = SparkSession.builder \
    .appName("HargaPangan-Analysis") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready:", spark.version)

Spark ready: 4.1.1



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
os.makedirs("../dashboard/data", exist_ok=True)

komoditas_list = [
    ("Beras Medium",  13500),
    ("Jagung",         6500),
    ("Kedelai",       14000),
    ("Gula Pasir",    16500),
    ("Minyak Goreng", 19000),
    ("Cabai Merah",   45000),
    ("Bawang Merah",  38000),
    ("Telur Ayam",    29000),
]

dummy_api = []
base_time = datetime.now()

for i in range(100):
    for nama, baseline in komoditas_list:
        harga      = baseline + random.gauss(0, baseline * 0.02)
        harga_prev = harga    + random.gauss(0, baseline * 0.01)
        ts         = (base_time + timedelta(minutes=i * 30)).isoformat()

        dummy_api.append({
            "komoditas":        nama,
            "harga":            round(harga, 0),
            "harga_sebelumnya": round(harga_prev, 0),
            "perubahan_persen": round((harga - harga_prev) / harga_prev * 100, 4),
            "harga_baseline":   baseline,
            "unit":             "kg",
            "timestamp":        ts,
            "sumber":           "simulator_bulog_baseline",
            "kota":             "Nasional",
        })

dummy_rss = [
    {"title": "Harga Cabai Merah Naik Jelang Akhir Bulan",
     "summary": "harga cabai naik di beberapa pasar tradisional jakarta",
     "link": "https://bisnis.com/1", "source": "bisnis.com",
     "published": datetime.now().isoformat()},

    {"title": "Bulog Pastikan Stok Beras Aman Tiga Bulan ke Depan",
     "summary": "stok beras nasional cukup dan tidak perlu khawatir",
     "link": "https://kompas.com/1", "source": "kompas.com",
     "published": datetime.now().isoformat()},

    {"title": "Harga Minyak Goreng Stabil di Pasaran",
     "summary": "harga minyak goreng tidak mengalami kenaikan berarti minggu ini",
     "link": "https://bisnis.com/2", "source": "bisnis.com",
     "published": datetime.now().isoformat()},

    {"title": "Petani Bawang Merah Keluhkan Harga Jual Turun",
     "summary": "harga bawang merah di tingkat petani anjlok akibat panen raya",
     "link": "https://kompas.com/2", "source": "kompas.com",
     "published": datetime.now().isoformat()},

    {"title": "Harga Telur Ayam Naik Tipis di Pasar Tradisional",
     "summary": "kenaikan harga telur dipicu meningkatnya permintaan",
     "link": "https://bisnis.com/3", "source": "bisnis.com",
     "published": datetime.now().isoformat()},
]

with open("../dashboard/data/live_api.json", "w") as f:
    json.dump(dummy_api, f)

with open("../dashboard/data/live_rss.json", "w") as f:
    json.dump(dummy_rss, f)

print(f"live_api.json : {len(dummy_api)} records")
print(f"live_rss.json : {len(dummy_rss)} records")

live_api.json : 800 records
live_rss.json : 5 records


### LOAD DATA

In [12]:
LOCAL_API = "../dashboard/data/live_api.json"
LOCAL_RSS = "../dashboard/data/live_rss.json"

pdf_api = pd.read_json(LOCAL_API)
pdf_rss = pd.read_json(LOCAL_RSS)

print(f"data API : {len(pdf_api)} records")
print(f"data RSS : {len(pdf_rss)} records")
print()
print("Schema API:")
print(pdf_api.dtypes)
print("\nSample API data (5 baris):")
print(pdf_api.head())

data API : 800 records
data RSS : 5 records

Schema API:
komoditas                      str
harga                        int64
harga_sebelumnya             int64
perubahan_persen           float64
harga_baseline               int64
unit                           str
timestamp           datetime64[us]
sumber                         str
kota                           str
dtype: object

Sample API data (5 baris):
       komoditas  harga  harga_sebelumnya  perubahan_persen  harga_baseline  \
0   Beras Medium  13673             13679           -0.0427           13500   
1         Jagung   6365              6375           -0.1600            6500   
2        Kedelai  14058             13909            1.0726           14000   
3     Gula Pasir  16517             16470            0.2864           16500   
4  Minyak Goreng  18869             18862            0.0383           19000   

  unit                  timestamp                    sumber      kota  
0   kg 2026-04-30 10:18:44.642172  simu

In [13]:
df_volatilitas = pdf_api.groupby("komoditas").agg({
    "harga": ["max", "min", "mean", "count"],
    "perubahan_persen": "mean"
}).reset_index()

df_volatilitas.columns = ["komoditas", "harga_max", "harga_min", "harga_avg", "jumlah_data", "avg_perubahan_persen"]
df_volatilitas["volatilitas_pct"] = ((df_volatilitas["harga_max"] - df_volatilitas["harga_min"]) / df_volatilitas["harga_avg"] * 100).round(2)
df_volatilitas = df_volatilitas.sort_values("volatilitas_pct", ascending=False)

print(df_volatilitas.to_string())
print("""
Interpretasi:
- volatilitas tinggi -> harga banyak naik turun selama periode pemantauan
- volatilitas rendah -> harga stabil, biasanya karena ada regulasi HET
- komoditas dengan volatilitas > 3% perlu perhatian khusus dari Bulog
""")

       komoditas  harga_max  harga_min  harga_avg  jumlah_data  avg_perubahan_persen  volatilitas_pct
7     Telur Ayam      30714      27468   28915.15          100              0.111300            11.23
4         Jagung       6816       6104    6491.22          100              0.062891            10.97
1   Beras Medium      14036      12717   13440.07          100              0.023249             9.81
2    Cabai Merah      46807      42479   44855.61          100              0.097617             9.65
5        Kedelai      14756      13403   14015.69          100              0.065350             9.65
0   Bawang Merah      39557      35908   37941.31          100              0.003303             9.62
3     Gula Pasir      17251      15761   16537.91          100              0.091073             9.01
6  Minyak Goreng      19771      18153   18962.85          100              0.165755             8.53

Interpretasi:
- volatilitas tinggi -> harga banyak naik turun selama periode pema

In [14]:
pdf_api_copy = pdf_api.copy()
pdf_api_copy['periode'] = pdf_api_copy['timestamp'].astype(str).str[:13]

df_tren = pdf_api_copy[pdf_api_copy['harga'].notna() & pdf_api_copy['komoditas'].notna()] \
    .groupby(['komoditas', 'periode']).agg({
        'harga': ['mean', 'min', 'max'],
        'perubahan_persen': 'mean'
    }).reset_index()

df_tren.columns = ["komoditas", "periode", "harga_rata", "harga_min", "harga_max", "avg_perubahan_pct"]
df_tren = df_tren.astype({
    "harga_rata": int,
    "harga_min": int, 
    "harga_max": int
}).sort_values(['komoditas', 'periode'])

print(df_tren.head(20).to_string())

print("\n--- ringkasan status tren ---")
summary = pdf_api.groupby('komoditas')['perubahan_persen'].agg(['mean', 'count']).reset_index()
summary.columns = ['komoditas', 'avg_perubahan_pct', 'jumlah_data']
summary['status_tren'] = summary['avg_perubahan_pct'].apply(
    lambda x: 'Naik' if x > 0.5 else ('Turun' if x < -0.5 else 'Stabil')
)
summary['avg_perubahan_pct'] = summary['avg_perubahan_pct'].round(4)
summary = summary.sort_values('avg_perubahan_pct', ascending=False)
print(summary.to_string())

print("""
Interpretasi:
- avg_perubahan_pct positif -> harga cenderung naik selama periode ini
- avg_perubahan_pct negatif -> harga cenderung turun
- data per jam berguna untuk menentukan kapan operasi pasar perlu dilakukan
""")

       komoditas        periode  harga_rata  harga_min  harga_max  avg_perubahan_pct
0   Bawang Merah  2026-04-30 10       37749      37405      38094            0.25595
1   Bawang Merah  2026-04-30 11       38182      37960      38405           -0.63890
2   Bawang Merah  2026-04-30 12       38308      37632      38985           -0.58460
3   Bawang Merah  2026-04-30 13       37687      36416      38959           -0.51755
4   Bawang Merah  2026-04-30 14       37117      37100      37134            0.17140
5   Bawang Merah  2026-04-30 15       38041      37801      38281            0.11775
6   Bawang Merah  2026-04-30 16       38438      38327      38549           -0.65815
7   Bawang Merah  2026-04-30 17       37826      37493      38159            0.60375
8   Bawang Merah  2026-04-30 18       37423      37092      37754           -0.04000
9   Bawang Merah  2026-04-30 19       37846      37816      37876           -1.68195
10  Bawang Merah  2026-04-30 20       37449      37308      37591

In [15]:
KEYWORDS = [
    ("Beras Medium",  ["beras", "padi"]),
    ("Jagung",        ["jagung"]),
    ("Kedelai",       ["kedelai", "tahu", "tempe"]),
    ("Gula Pasir",    ["gula"]),
    ("Minyak Goreng", ["minyak goreng", "minyak"]),
    ("Cabai Merah",   ["cabai", "cabe"]),
    ("Bawang Merah",  ["bawang merah", "bawang"]),
    ("Telur Ayam",    ["telur"]),
]

hasil_korelasi = []

for komoditas, keywords in KEYWORDS:
    # Hitung frekuensi berita
    mask = pdf_rss[['title', 'summary']].astype(str).apply(
        lambda x: any(kw.lower() in x.str.lower().values for kw in keywords), axis=1
    )
    frekuensi = mask.sum()
    
    # Hitung avg change dari API
    avg_change = pdf_api[pdf_api['komoditas'] == komoditas]['perubahan_persen'].mean()
    avg_change = round(avg_change, 4) if pd.notna(avg_change) else 0.0
    
    # Tentukan label interpretasi
    if frekuensi > 5 and avg_change > 0.3:
        label = "Berita tinggi + harga naik"
    elif frekuensi > 5 and avg_change < -0.3:
        label = "Berita tinggi + harga turun"
    elif frekuensi > 5:
        label = "Banyak berita, harga stabil"
    elif avg_change > 0.5:
        label = "Harga naik, sedikit berita"
    else:
        label = "Normal"
    
    hasil_korelasi.append({
        "komoditas": komoditas,
        "frekuensi_berita": int(frekuensi),
        "avg_perubahan_persen": avg_change,
        "interpretasi": label,
    })

df_korelasi = pd.DataFrame(hasil_korelasi).sort_values('frekuensi_berita', ascending=False)
print(df_korelasi.to_string(index=False))

print("""
Interpretasi:
- harga naik + berita banyak -> kemungkinan ada faktor eksternal seperti cuaca atau impor
- harga naik tapi sedikit berita -> sistem ini berfungsi sebagai early warning
- berita seringkali terlambat dibanding pergerakan harga aktual
""")

    komoditas  frekuensi_berita  avg_perubahan_persen interpretasi
 Beras Medium                 0                0.0232       Normal
       Jagung                 0                0.0629       Normal
      Kedelai                 0                0.0654       Normal
   Gula Pasir                 0                0.0911       Normal
Minyak Goreng                 0                0.1658       Normal
  Cabai Merah                 0                0.0976       Normal
 Bawang Merah                 0                0.0033       Normal
   Telur Ayam                 0                0.1113       Normal

Interpretasi:
- harga naik + berita banyak -> kemungkinan ada faktor eksternal seperti cuaca atau impor
- harga naik tapi sedikit berita -> sistem ini berfungsi sebagai early warning
- berita seringkali terlambat dibanding pergerakan harga aktual



In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

hasil_mlllib = {}

for target in ["Beras Medium", "Cabai Merah", "Telur Ayam"]:
    df_filtered = pdf_api[pdf_api['komoditas'] == target][['harga', 'timestamp']].copy()
    df_filtered = df_filtered.dropna()
    df_filtered = df_filtered.sort_values('timestamp').reset_index(drop=True)
    
    count = len(df_filtered)
    print(f"\n{target} ({count} records)")
    
    if count < 5:
        print("  data belum cukup, skip")
        hasil_mlllib[target] = {"status": "insufficient_data"}
        continue
    
    # Buat feature idx
    X = np.arange(count).reshape(-1, 1)
    y = df_filtered['harga'].values
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Fit model
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    
    # Predict dan evaluate
    y_pred = lr.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    koef = lr.coef_[0]
    tren = "naik" if koef > 0 else ("turun" if koef < 0 else "stabil")
    
    print(f"  koefisien : {koef:.4f}  ({tren})")
    print(f"  R²        : {r2:.4f}")
    print(f"  RMSE      : {rmse:.2f}")
    
    hasil_mlllib[target] = {
        "komoditas": target,
        "koefisien": round(koef, 4),
        "intercept": round(lr.intercept_, 2),
        "r2": round(r2, 4),
        "rmse": round(rmse, 2),
        "tren": tren,
    }

print("""
Interpretasi:
- koefisien > 0 -> harga diprediksi naik per satuan waktu
- R² mendekati 1 -> model akurat
- R² mendekati 0 -> harga bergerak terlalu acak untuk diprediksi secara linear
""")


Beras Medium (100 records)
  koefisien : 0.3626  (naik)
  R²        : -0.0310
  RMSE      : 268.91

Cabai Merah (100 records)
  koefisien : -2.0740  (turun)
  R²        : -0.0490
  RMSE      : 813.29

Telur Ayam (100 records)
  koefisien : 2.8167  (naik)
  R²        : -0.0820
  RMSE      : 649.60

Interpretasi:
- koefisien > 0 -> harga diprediksi naik per satuan waktu
- R² mendekati 1 -> model akurat
- R² mendekati 0 -> harga bergerak terlalu acak untuk diprediksi secara linear



In [17]:
OUTPUT_FILE = "../dashboard/data/spark_results.json"
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

hasil_dashboard = {
    "generated_at": datetime.now().isoformat(),
    "volatilitas": df_volatilitas.to_dict('records'),
    "tren_harga": df_tren.to_dict('records'),
    "korelasi_berita": hasil_korelasi,
    "prediksi_mlllib": hasil_mlllib,
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(hasil_dashboard, f, ensure_ascii=False, indent=2, default=str)

print("spark_results.json tersimpan")
print(f"  volatilitas  : {len(hasil_dashboard['volatilitas'])} komoditas")
print(f"  tren_harga   : {len(hasil_dashboard['tren_harga'])} baris")
print(f"  korelasi     : {len(hasil_dashboard['korelasi_berita'])} komoditas")
print(f"  mlllib       : {len(hasil_dashboard['prediksi_mlllib'])} model")

print("\nSelesai! (tidak stop spark karena masih bisa digunakan di cell lain jika diperlukan)")

spark_results.json tersimpan
  volatilitas  : 8 komoditas
  tren_harga   : 400 baris
  korelasi     : 8 komoditas
  mlllib       : 3 model

Selesai! (tidak stop spark karena masih bisa digunakan di cell lain jika diperlukan)
